# Apply BIDS Events Cleanup for OpenNeuro Feedback

This notebook is the processing/write-back step. Run it only after reviewing:

- `01_qc_openneuro_events.ipynb`
- `02_check_stimulus_id_audio_alignment.ipynb`

It can remove `raw_trial_type`, update WAV-based `duration`, and add `stimulus_id` metadata to `events.json`. It backs up every edited file before writing.


In [6]:
from pathlib import Path
from datetime import datetime
import json
import shutil
import wave

import pandas as pd

BIDS_ROOT = Path('/Users/yanyuwoo/Data/bids')
STIMULI_DIR = BIDS_ROOT / 'stimuli'
PROJECT_DATA_ROOT = Path('/Users/yanyuwoo/Data/Alice Comprehension')
QC_DIR = PROJECT_DATA_ROOT / 'qc' / 'openneuro_events_feedback'
BACKUP_ROOT = PROJECT_DATA_ROOT / 'intermediate' / 'openneuro_events_feedback_backups'

# Keep False until the dry-run reports look correct.

QC_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

print(f'BIDS root: {BIDS_ROOT}')
print(f'Stimuli dir: {STIMULI_DIR}')


BIDS root: /Users/yanyuwoo/Data/bids
Stimuli dir: /Users/yanyuwoo/Data/bids/stimuli


## Processing Guard

Default is dry-run. Set `APPLY_CHANGES = True` only after stimulus IDs are confirmed.

In [7]:
# Keep False until QC reports and audio-alignment checks are reviewed.
APPLY_CHANGES = True
print(f'Apply changes: {APPLY_CHANGES}')


Apply changes: True


## Gather Files and WAV Durations

Durations are read from the actual WAV files so the notebook does not depend on hard-coded segment lengths.

In [8]:
def subject_from_events_path(path):
    return path.parts[-3]


def read_wav_duration(path):
    with wave.open(str(path), 'rb') as wav:
        return wav.getnframes() / wav.getframerate()


events_paths = sorted(
    BIDS_ROOT.glob('sub-*/eeg/sub-*_task-alice_events.tsv'),
    key=lambda p: int(subject_from_events_path(p).split('-')[1]),
)
events_json_paths = [path.with_suffix('.json') for path in events_paths]

wav_durations = {
    int(path.stem): read_wav_duration(path)
    for path in sorted(STIMULI_DIR.glob('*.wav'), key=lambda p: int(p.stem))
}

duration_table = pd.DataFrame(
    [{'stimulus_id': key, 'wav_duration': value} for key, value in wav_durations.items()]
)

print(f'Events files: {len(events_paths)}')
print(f'WAV files: {len(wav_durations)}')
display(duration_table)

Events files: 49
WAV files: 12


,stimulus_id,wav_duration
0,1,57.540612
1,2,60.845193
2,3,63.259433
3,4,69.988571
4,5,66.272540
5,6,63.777551
6,7,62.896848
7,8,57.310612
8,9,57.226145
9,10,61.269660


## 4. Prepare `events.json` Metadata

This adds documentation for `stimulus_id` and clarifies that `value` is the raw trigger code, not the WAV segment identity.

In [9]:
STIMULUS_ID_METADATA = {
    'Description': 'Identifier of the Alice audio segment presented at this event. This refers to the WAV stimulus file in the stimuli directory, not the raw trigger code.',
    'Levels': {str(i): f'Alice chapter-one audio segment {i}; stimulus file {i}.wav' for i in sorted(wav_durations)},
}

EVENTS_JSON_UPDATES = {
    'duration': {
        'Description': 'Duration of the presented audio stimulus in seconds, measured from the corresponding WAV file.',
        'Units': 's',
    },
    'value': {
        'Description': 'Raw trigger code associated with the event. This is preserved from the recording system and should not be assumed to equal stimulus_id.',
    },
    'trial_type': {
        'Description': 'Standardized stimulus label derived from the event marker, using Stimulus/1 through Stimulus/12.',
    },
    'stimulus_id': STIMULUS_ID_METADATA,
}

example_json_path = events_json_paths[0]
with example_json_path.open(encoding='utf-8') as f:
    example_json = json.load(f)
preview_json = {**example_json, **EVENTS_JSON_UPDATES}

print(f'Example JSON path: {example_json_path}')
print(json.dumps(preview_json, indent=2)[:2000])

Example JSON path: /Users/yanyuwoo/Data/bids/sub-01/eeg/sub-01_task-alice_events.json
{
  "onset": {
    "Description": "Onset (in seconds) of the event from the beginning of the first datapoint. Negative onsets account for events before the first stored data point.",
    "Units": "s"
  },
  "duration": {
    "Description": "Duration of the presented audio stimulus in seconds, measured from the corresponding WAV file.",
    "Units": "s"
  },
  "sample": {
    "Description": "The event onset time in number of sampling points.First sample is 0."
  },
  "value": {
    "Description": "Raw trigger code associated with the event. This is preserved from the recording system and should not be assumed to equal stimulus_id."
  },
  "trial_type": {
    "Description": "Standardized stimulus label derived from the event marker, using Stimulus/1 through Stimulus/12."
  },
  "stimulus_id": {
    "Description": "Identifier of the Alice audio segment presented at this event. This refers to the WAV stim

## What This Writes

When `APPLY_CHANGES=True`, this notebook writes to `/Users/yanyuwoo/Data/bids`:

- drops `raw_trial_type` from each `events.tsv`
- replaces trigger-pulse duration with WAV duration
- updates each `events.json` with `stimulus_id`, `value`, `duration`, and `trial_type` descriptions
- saves backups under `/Users/yanyuwoo/Data/Alice Comprehension/intermediate/openneuro_events_feedback_backups`


## 5. Apply Changes

This cell writes only when `APPLY_CHANGES = True`. It backs up every edited `events.tsv` and `events.json` first.

In [10]:
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
backup_dir = BACKUP_ROOT / timestamp
apply_rows = []

for events_path in events_paths:
    subject = subject_from_events_path(events_path)
    events_json_path = events_path.with_suffix('.json')

    df = pd.read_csv(events_path, sep='\t')
    original_columns = list(df.columns)

    removed_raw_trial_type = False
    if 'raw_trial_type' in df.columns:
        df = df.drop(columns=['raw_trial_type'])
        removed_raw_trial_type = True

    updated_duration = False
    if 'stimulus_id' in df.columns:
        new_duration = pd.to_numeric(
            df['stimulus_id'].map(lambda value: wav_durations.get(int(value)) if pd.notna(value) else None),
            errors='coerce',
        )
        old_duration = pd.to_numeric(df['duration'], errors='coerce')
        if not old_duration.round(6).equals(new_duration.round(6)):
            df['duration'] = new_duration
            updated_duration = True

    with events_json_path.open(encoding='utf-8') as f:
        events_json = json.load(f)
    updated_json = {**events_json, **EVENTS_JSON_UPDATES}
    json_changed = updated_json != events_json

    changed = removed_raw_trial_type or updated_duration or json_changed

    if changed and APPLY_CHANGES:
        subject_backup_dir = backup_dir / subject / 'eeg'
        subject_backup_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(events_path, subject_backup_dir / events_path.name)
        shutil.copy2(events_json_path, subject_backup_dir / events_json_path.name)

        df.to_csv(events_path, sep='\t', index=False)
        with events_json_path.open('w', encoding='utf-8') as f:
            json.dump(updated_json, f, indent=4)
            f.write('\n')

    apply_rows.append({
        'subject': subject,
        'changed': changed,
        'written': bool(changed and APPLY_CHANGES),
        'removed_raw_trial_type': removed_raw_trial_type,
        'updated_duration': updated_duration,
        'updated_events_json': json_changed,
        'original_columns': ';'.join(original_columns),
        'final_columns': ';'.join(df.columns),
    })

apply_report = pd.DataFrame(apply_rows)
suffix = 'applied' if APPLY_CHANGES else 'dry_run'
apply_report_path = QC_DIR / f'openneuro_events_feedback_{suffix}.csv'
apply_report.to_csv(apply_report_path, index=False)

print(f'Wrote apply report: {apply_report_path}')
if APPLY_CHANGES:
    print(f'Backups: {backup_dir}')
else:
    print('Dry run only. Set APPLY_CHANGES=True to write changes.')

display(apply_report.head())
display(apply_report[['changed', 'written', 'removed_raw_trial_type', 'updated_duration', 'updated_events_json']].sum())

Wrote apply report: /Users/yanyuwoo/Data/Alice Comprehension/qc/openneuro_events_feedback/openneuro_events_feedback_applied.csv
Backups: /Users/yanyuwoo/Data/Alice Comprehension/intermediate/openneuro_events_feedback_backups/20260707-160630


,subject,changed,written,removed_raw_trial_type,updated_duration,updated_events_json,original_columns,final_columns
0,sub-01,True,True,True,True,True,onset;duration;trial_type;value;sample;raw_tri...,onset;duration;trial_type;value;sample;stimulu...
1,sub-02,True,True,True,True,True,onset;duration;trial_type;value;sample;raw_tri...,onset;duration;trial_type;value;sample;stimulu...
2,sub-03,True,True,True,True,True,onset;duration;trial_type;value;sample;raw_tri...,onset;duration;trial_type;value;sample;stimulu...
3,sub-04,True,True,True,True,True,onset;duration;trial_type;value;sample;raw_tri...,onset;duration;trial_type;value;sample;stimulu...
4,sub-05,True,True,True,True,True,onset;duration;trial_type;value;sample;raw_tri...,onset;duration;trial_type;value;sample;stimulu...


changed                   49
written                   49
removed_raw_trial_type    49
updated_duration          49
updated_events_json       49
dtype: int64